## Install requrired version of modules

In [2]:
!pip install -q langchain_openai==0.3.33 langchain==0.3.27 langchain-community==0.3.24

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.8/450.8 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 65.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.5 requires langchain-core>=1.0.0, but you have langchain-core 0.3.80 which is incompatible.


## Load API Keys

In [5]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
import warnings

warnings.filterwarnings("ignore")

## Simple Example of Langchain Conversation Chain

In [3]:
from langchain import OpenAI, ConversationChain

In [9]:
llm = OpenAI(temperature=0)
conversation = ConversationChain(llm=llm)

In [10]:
response1 = conversation.invoke("Who won the 2018 FIFA world cup?")
response1

{'input': 'Who won the 2018 FIFA world cup?',
 'history': '',
 'response': " The 2018 FIFA World Cup was won by France. They defeated Croatia 4-2 in the final match. This was France's second World Cup victory, with their first being in 1998. The tournament was held in Russia and lasted from June 14 to July 15, 2018. France's star player, Kylian Mbappé, was named the Best Young Player of the tournament."}

In [11]:
response2 = conversation.invoke("Who won the next?")
response2

{'input': 'Who won the next?',
 'history': "Human: Who won the 2018 FIFA world cup?\nAI:  The 2018 FIFA World Cup was won by France. They defeated Croatia 4-2 in the final match. This was France's second World Cup victory, with their first being in 1998. The tournament was held in Russia and lasted from June 14 to July 15, 2018. France's star player, Kylian Mbappé, was named the Best Young Player of the tournament.",
 'response': ' The next FIFA World Cup will be held in 2022 in Qatar. The winner of the 2022 World Cup has not yet been determined, as the qualifying rounds are still ongoing. However, the current top-ranked team in the world is Belgium, followed by France and Brazil.'}

In [12]:
conversation2 = ConversationChain(llm=llm)

In [13]:
response1 = conversation2.invoke("what is my name?")
response1

{'input': 'what is my name?',
 'history': '',
 'response': ' I am sorry, I do not have access to your personal information. I am an AI and do not have the ability to know your name unless you tell me. Is there something else you would like to know?'}

In [14]:
response2 = conversation2.invoke("what is the date today?")
response2

{'input': 'what is the date today?',
 'history': 'Human: what is my name?\nAI:  I am sorry, I do not have access to your personal information. I am an AI and do not have the ability to know your name unless you tell me. Is there something else you would like to know?',
 'response': ' Today is Tuesday, October 12th, 2021. Is there anything else you would like to know?'}

## Agents that use Tools

In [11]:
from langchain.tools import Tool
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.prompts import PromptTemplate
from datetime import datetime

In [13]:
import langchain
langchain.__version__

'0.3.27'

In [16]:
today_tool = Tool(
    name="Date Today",
    func=lambda x:datetime.now().strftime("%Y-%m-%d"),
    description="Returns the current date",
)

In [17]:
template = """You are a helpful assistant who can use tools to answer user's questions.
You can use the Date Today tool to get the current date. Question: {question}"""

prompt = PromptTemplate(
    template=template,
    input_variables=["question"],
)

In [18]:
llm = ChatOpenAI(temperature=0)

agent = initialize_agent(
    tools=[today_tool],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=False
)

In [19]:
agent.run(prompt.format(question='what is the date today?'))

'The date today is 2025-12-06.'

In [20]:
agent.run(prompt.format(question='how many days to new year?'))

'There are 26 days left until the new year.'

In [21]:
agent = initialize_agent(
    tools=[today_tool],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

In [22]:
agent.run(prompt.format(question='what is the date today?'))



> Entering new AgentExecutor chain...
I should use the Date Today tool to get the current date.
Action: Date Today
Action Input: 
Observation: 2025-12-06
Thought:I now know the final answer
Final Answer: The date today is 2025-12-06.

> Finished chain.


'The date today is 2025-12-06.'

In [23]:
agent.run(prompt.format(question='what is the current year?'))



> Entering new AgentExecutor chain...
I should use the Date Today tool to get the current year.
Action: Date Today
Action Input: x
Observation: 2025-12-06
Thought:The current year is 2025.
Final Answer: 2025

> Finished chain.


'2025'

In [24]:
agent.run(prompt.format(question='what is the current week?'))



> Entering new AgentExecutor chain...
I should use the Date Today tool to get the current date and then determine the week number.
Action: Date Today
Action Input: 
Observation: 2025-12-06
Thought:Now I need to determine the week number for the current date.
Action: Week Number
Action Input: 2025-12-06
Observation: Week Number is not a valid tool, try one of [Date Today].
Thought:I should try a different approach to determine the current week number.
Action: Calculate Week Number
Action Input: 2025-12-06
Observation: Calculate Week Number is not a valid tool, try one of [Date Today].
Thought:I can simply count the number of weeks that have passed in the current year.
Action: Calculate Weeks Passed
Action Input: 2025-12-06
Observation: Calculate Weeks Passed is not a valid tool, try one of [Date Today].
Thought:I can calculate the week number manually by dividing the day of the year by 7.
Action: Calculate Week Number Manually
Action Input: 2025-12-06
Observation: Calculate Week Numbe

'I cannot determine the current week number using the tools available.'

In [7]:
today_tool = Tool(
    name="Date Today",
    func=lambda x:datetime.now().strftime("%Y-%m-%d"),
    description=(
        "Returns today's date in ISO format (YYYY-MM-DD). "
        "Always use this tool for any question about the current date, "
        "including questions about the current year, month, day, weekday, "
        "or the current week number. After getting the date, use your own "
        "reasoning to compute the requested information, such as the week "
        "number of the year."
    ),
)

In [8]:
template = """You are a helpful assistant who can use tools to answer user's questions.
You can use the Date Today tool to get the current date. Question: {question}"""

prompt = PromptTemplate(
    template=template,
    input_variables=["question"],
)

In [9]:
llm = ChatOpenAI(temperature=0)

agent = initialize_agent(
    tools=[today_tool],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=False
)

/tmp/ipython-input-2475611523.py:3: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(


In [12]:
agent.run(prompt.format(question='what is the current week?'))

'The current week number is 49.'

## Multi-Tool Agent

In [17]:
!pip install -q serpapi google-search-results

  Preparing metadata (setup.py) ... done


In [14]:
# Core imports for LLM and agents
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, load_tools
# PromptTemplate for constructing the final prompt string
from langchain.prompts import PromptTemplate

In [15]:
llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")

In [18]:
# 2. Load tools:
#    - "serpapi": web search
#    - "llm-math": math through the LLM
tools = load_tools(
    ["serpapi", "llm-math"],
    llm=llm
)

In [19]:
# 3. Create an agent that uses ReAct-style reasoning with descriptions
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent="zero-shot-react-description",
    verbose=False
)

In [20]:
# Here we define a PromptTemplate that:
# - Explains to the LLM that it has access to a search tool and a calculator.
# - Instructs it to first use search, then do math with the result.
# - Inserts the user question via {question}.

prompt_template = PromptTemplate(
    input_variables=["question"],
    template="""
You are an AI assistant that can use tools to answer questions.

You have access to:
1. A web search tool (serpapi) to look up real-world facts.
2. A math tool (llm-math) to perform precise numeric calculations.

For the given task, follow these steps:
- First, use the web search tool to find any factual information you need.
- Then, use the math tool to perform any required calculations.
- Finally, explain your reasoning and provide a clear final answer.

Now, here is the task you must solve:
{question}
""".strip()
)

In [21]:
# Define a realistic, simple question that needs BOTH tools:
# - Use search to find Google's founding year.
# - Use math to add 25 to that year.

question_text = """
Find the year Google was founded, then add 25 to that year.
Show the calculation and the final result.
""".strip()

In [22]:
# Use the PromptTemplate to create the final prompt string
formatted_prompt = prompt_template.format(question=question_text)

# Pass the formatted prompt STRING to the agent
result = agent.run(formatted_prompt)

# Print the final answer from the agent
print("\n===== FINAL ANSWER =====\n")
print(result)


===== FINAL ANSWER =====

Google was founded in 1998, so adding 25 to that year gives us the year 2023.


In [23]:
question_text = """
Find the current CEO of Google, then add 25 to the length of the first Name.
Show the calculation and the final result.
""".strip()

In [24]:
# Use the PromptTemplate to create the final prompt string
formatted_prompt = prompt_template.format(question=question_text)

# Pass the formatted prompt STRING to the agent
result = agent.run(formatted_prompt)

# Print the final answer from the agent
print("\n===== FINAL ANSWER =====\n")
print(result)

ValueError: LLMMathChain._evaluate("
len("Sundar") + 25
") raised error: 'VariableNode' object is not callable. Please try again with a valid numerical expression

In [25]:
prompt_template = PromptTemplate(
    input_variables=["question"],
    template="""
You are an AI assistant that can use tools to answer questions.

You have access to:
1. A web search tool (serpapi) to look up real-world facts.
2. A math tool (llm-math) to perform precise numeric calculations.

IMPORTANT RULES:
- Use the web search tool for real-world information (like names, dates, etc.).
- Do NOT use the math tool to manipulate strings or compute lengths of words.
- If you need the length of a name or word, count the letters yourself in your reasoning.
- Use the math tool ONLY on simple numeric expressions (like 25 + 6, 100 / 4, etc.).

For the given task, follow these steps:
1. Use the web search tool to find any facts you need.
2. In your own reasoning (NOT using the math tool), figure out any text-related details,
   for example, count how many letters are in a given first name.
3. Only after you have a plain number, use the math tool to perform the final arithmetic.
4. Finally, explain your reasoning and provide a clear final answer.

Now, here is the task you must solve:
{question}
""".strip()
)

In [29]:
# 3. Create an agent that uses ReAct-style reasoning with descriptions
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent="zero-shot-react-description",
    verbose=True
)

In [30]:
# Use the PromptTemplate to create the final prompt string
formatted_prompt = prompt_template.format(question=question_text)

# Pass the formatted prompt STRING to the agent
result = agent.run(formatted_prompt)

# Print the final answer from the agent
print("\n===== FINAL ANSWER =====\n")
print(result)



> Entering new AgentExecutor chain...
I need to find the current CEO of Google first before I can add 25 to the length of their first name.
Action: Search
Action Input: "current CEO of Google"
Observation: Sundar Pichai
Thought:The first name "Sundar" has 6 letters, so I need to add 25 to 6.
Action: Calculator
Action Input: 25 + 6
Observation: Answer: 31
Thought:I now know the final answer.
Final Answer: 31

> Finished chain.

===== FINAL ANSWER =====

31


## Wiki Tool

In [31]:
!pip install -q wikipedia

  Preparing metadata (setup.py) ... done


In [32]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.tools import Tool

In [33]:
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

wikipedia_tool = Tool(
    name="Wikipedia",
    func=wiki.run,
    description="Useful for fetching facts from Wikipedia"
)

In [34]:
query_1 = "Get me the top 5 Machine learning algorithms."
result_1 = wikipedia_tool.run(query_1)
result_1

'Page: Adversarial machine learning\nSummary: Adversarial machine learning is the study of the attacks on machine learning algorithms, and of the defenses against such attacks. A survey from May 2020 revealed practitioners\' common feeling for better protection of machine learning systems in industrial applications.\nMachine learning techniques are mostly designed to work on specific problem sets, under the assumption that the training and test data are generated from the same statistical distribution (IID). However, this assumption is often dangerously violated in practical high-stake applications, where users may intentionally supply fabricated data that violates the statistical assumption.\nMost common attacks in adversarial machine learning include evasion attacks, data poisoning attacks, Byzantine attacks and model extraction.\n\nPage: Prompt engineering\nSummary: Prompt engineering is the process of structuring or crafting an instruction in order to produce better outputs from a 

## Custom Function Tool

In [35]:
def preprocess_string(x):
    if len(x) < 10:
        return "Too short to process"

    return x.strip().strip("```").strip('json')

In [37]:
string_preprocessor_tool = Tool(
    name="string preprocessor",
    func=lambda x: preprocess_string(x),
    description="Useful for doing string pre-processing"
)

In [38]:
query_3 = '{"age": 30}'
result_3 = string_preprocessor_tool.run(query_3)
result_3

'{"age": 30}'

In [39]:
query_3 = '{"e": 30}'
result_3 = string_preprocessor_tool.run(query_3)
result_3

'Too short to process'

## Stuctured Output

In [40]:
from openai import OpenAI
from langchain.prompts import PromptTemplate
import json
import os

In [42]:
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_API_BASE")
)

In [43]:
json_template = PromptTemplate(
    input_variables=["text"],
    template="""
Extract key insights from the text below.
Return the result strictly in JSON with keys:
- "summary"
- "keywords" (as a list)
- "sentiment" (positive/neutral/negative)

Text:
{text}

Return only valid JSON. No explanations.
"""
)

In [44]:
sample_text = """
Python is widely used for data science because of its rich ecosystem
of libraries such as NumPy, Pandas, and Scikit-learn.
It is also easy to learn and has a strong community.
"""

In [45]:
prompt_json = json_template.format(text=sample_text)

In [46]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt_json}]
)

In [47]:
response

ChatCompletion(id='chatcmpl-CkBnKnEeR8ueJgYIYm8rkpkfsBIpx', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "summary": "Python is popular for data science due to its extensive libraries and ease of learning, along with a strong community support.",\n  "keywords": ["Python", "data science", "libraries", "NumPy", "Pandas", "Scikit-learn", "easy to learn", "community"],\n  "sentiment": "positive"\n}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1765124462, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_aa07c96156', usage=CompletionUsage(completion_tokens=74, prompt_tokens=101, total_tokens=175, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)

In [48]:
text_output = response.choices[0].message.content
text_output

'{\n  "summary": "Python is popular for data science due to its extensive libraries and ease of learning, along with a strong community support.",\n  "keywords": ["Python", "data science", "libraries", "NumPy", "Pandas", "Scikit-learn", "easy to learn", "community"],\n  "sentiment": "positive"\n}'

In [49]:
try:
    parsed_output = json.loads(text_output)
    print("Parsed JSON:\n", parsed_output)

    print("\nSummary:", parsed_output["summary"])
    print("Keywords:", parsed_output["keywords"])
    print("Sentiment:", parsed_output["sentiment"])

except json.JSONDecodeError:
    print("Model returned invalid JSON. Attempting to fix...")

    # Optional: cleaning hack (simple, not robust)
    cleaned = text_output.strip().strip("```").strip('json')
    try:
        parsed_output = json.loads(cleaned)
        print("Fixed JSON:\n", parsed_output)
    except:
        print("Could not parse JSON. Manual review needed.")

Parsed JSON:
 {'summary': 'Python is popular for data science due to its extensive libraries and ease of learning, along with a strong community support.', 'keywords': ['Python', 'data science', 'libraries', 'NumPy', 'Pandas', 'Scikit-learn', 'easy to learn', 'community'], 'sentiment': 'positive'}

Summary: Python is popular for data science due to its extensive libraries and ease of learning, along with a strong community support.
Keywords: ['Python', 'data science', 'libraries', 'NumPy', 'Pandas', 'Scikit-learn', 'easy to learn', 'community']
Sentiment: positive
